# Experiment 1A — Exact Paper Reproduction (Leakage Demo)

**Goal:** Reproduce the paper's methodology *exactly* to replicate their claimed ~99% accuracy.

| Setting | Value | Paper Reference |
|---|---|---|
| Input | RGB 3-channel, 224×224 | Section III-C, Fig. 4 |
| Preprocessing | CLAHE (clip=2.0, tile=4×4) | Section III-B, Fig. 3 |
| Split | 80/20 slice-level random | Section III-B |
| Augmentation | Rotation 20°, translate 0.1, shear 10°, zoom 0.9-1.1, H-flip | Section III-B |
| Model | CNN_Model1 (deep) + CNN_Model2 (shallow) + MHA×2 + MLP | Section III-C, Fig. 4-6 |
| Optimizer | Adam, lr=1e-3 | Table 5 |
| Loss | Sparse Categorical Cross-Entropy | Table 5 |
| Epochs | 50, batch_size=32 | Table 5 |
| MHA Config | heads=4, key_dim=16, value_dim=64 | Table 5, Table 7 |

**⚠️ This experiment reproduces the DATA LEAKAGE in the paper.**  
Slice-level random split scatters slices from the same patient across train/test → inflated metrics.

> **Do NOT use these results as honest baselines.** This exists solely to demonstrate the leakage effect.


---
## 0 · Configuration & Imports

In [ ]:
import os, sys, json, random, csv, re, time
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, roc_curve, auc
)
from sklearn.preprocessing import label_binarize
from PIL import Image
import cv2

# ── Paths ────────────────────────────────────────────────────────────────────
DATASET_ROOT = "/home/metal/Experiment/data/The IQ-OTHNCCD lung cancer dataset"
RESULTS_DIR  = Path("./results")
RESULTS_DIR.mkdir(exist_ok=True)

# ── Dataset ──────────────────────────────────────────────────────────────────
CLASS_FOLDERS  = ["Benign cases", "Malignant cases", "Normal cases"]
CLASS_TO_LABEL = {"Benign cases": 0, "Malignant cases": 1, "Normal cases": 2}
LABEL_TO_CLASS = {0: "Benign",    1: "Malignant",      2: "Normal"}
NUM_CLASSES    = 3

# ── Paper-matched hyperparameters (Table 5) ─────────────────────────────────
EPOCHS       = 50
BATCH_SIZE   = 32
LR           = 1e-3          # Adam with lr=0.001 (paper Table 5)
WEIGHT_DECAY = 0.0           # paper does not mention weight decay
GRAD_CLIP    = 1.0
WARMUP_EPOCHS= 0             # paper does not mention warmup
EARLY_STOP_PAT = 999         # paper trains full 50 epochs
NUM_WORKERS  = 4
IMG_SIZE     = 224

# Paper split: 80/20 (no validation set mentioned explicitly; we create 80/10/10)
TRAIN_RATIO  = 0.80
VAL_RATIO    = 0.10          # small val for early-stop monitoring

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {DEVICE}")
print(f"Dataset: {DATASET_ROOT}")
print(f"Exists : {Path(DATASET_ROOT).exists()}")

---
## 1 · Model Definition — Exact Paper Architecture (CCAL)

From the paper Section III-C and Figures 4, 5, 6:

```
Input (3×224×224)  ─── RGB, CLAHE-preprocessed
    ├─ CNN_Model1 ─ 6 conv layers (32→64→64→128→128→256) + 3 MaxPool + GlobalMaxPool ──► (B, 256)
    └─ CNN_Model2 ─ 2 conv layers (16→32) + 2 MaxPool + GlobalMaxPool ──────────────────► (B,  32)
                         concat → (B, 288)
                         Linear → (B, T×E)  reshape → (B, T, E)
                         MHA × 2  (heads=4, key_dim=16)  → (B, T, E)
                         GlobalAvgPool1D    → (B, E)
                         BN → Dense(256) → GELU → BN → Dropout(0.3) → Dense(3)
                         Logits + attention_weights
```

**Note on parameter count:** The paper claims 14.27M parameters (Table 10),
but the described architecture yields ~0.8-1.3M. See our architecture analysis
for discussion of this discrepancy.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CNN_Model1 : deep branch  → (B, 256)
# 6 conv layers, 3 MaxPool, 1 GlobalMaxPool
# Channel progression: 3 → 32 → 64 → 64 → 128 → 128 → 256
# Paper: Section III-C, Figure 4 (left branch)
# ─────────────────────────────────────────────────────────────────────────────
class CNN_Model1(nn.Module):
    """Deep branch: extracts hierarchical, complex patterns
    (textures, irregular nodule boundaries, structural anomalies)."""
    def __init__(self, inp_chn=3, out_chn=32):
        super().__init__()
        self.conv1 = nn.Conv2d(inp_chn,       out_chn,   kernel_size=3, padding=1)
        self.relu  = nn.ReLU()
        self.pool  = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(out_chn,       out_chn*2, kernel_size=3, padding=1)  # 64
        self.conv3 = nn.Conv2d(out_chn*2,     out_chn*2, kernel_size=3, padding=1)  # 64
        self.conv4 = nn.Conv2d(out_chn*2,     out_chn*4, kernel_size=3, padding=1)  # 128
        self.conv5 = nn.Conv2d(out_chn*4,     out_chn*4, kernel_size=3, padding=1)  # 128
        self.conv6 = nn.Conv2d(out_chn*4,     out_chn*8, kernel_size=3, padding=1)  # 256
        self.global_max_pool = nn.AdaptiveMaxPool2d(1)  # → (B, 256, 1, 1)

    def forward(self, x):
        x = self.relu(self.conv1(x))   # 224×224, 32ch
        x = self.pool(x)               # 112×112
        x = self.relu(self.conv2(x))   # 112×112, 64ch
        x = self.relu(self.conv3(x))   # 112×112, 64ch
        x = self.pool(x)               # 56×56
        x = self.relu(self.conv4(x))   # 56×56,  128ch
        x = self.relu(self.conv5(x))   # 56×56,  128ch
        x = self.pool(x)               # 28×28
        x = self.relu(self.conv6(x))   # 28×28,  256ch
        x = self.global_max_pool(x)    # (B, 256, 1, 1)
        return x                       # kept 4-D; MainModel flattens


# ─────────────────────────────────────────────────────────────────────────────
# CNN_Model2 : shallow branch → (B, 32)
# 2 conv layers, 2 MaxPool, 1 GlobalMaxPool
# Channel progression: 3 → 16 → 32
# Paper: Section III-C, Figure 4 (right branch)
# ─────────────────────────────────────────────────────────────────────────────
class CNN_Model2(nn.Module):
    """Shallow branch: captures broader, generalized features
    (shape, size, overall density variations)."""
    def __init__(self, inp_chn=3, out_chn=16):
        super().__init__()
        self.conv1 = nn.Conv2d(inp_chn,   out_chn,   kernel_size=3, padding=1)  # 16
        self.relu  = nn.ReLU()
        self.pool  = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(out_chn,   out_chn*2, kernel_size=3, padding=1)  # 32
        self.global_max_pool = nn.AdaptiveMaxPool2d(1)  # → (B, 32, 1, 1)

    def forward(self, x):
        x = self.relu(self.conv1(x))   # 224×224, 16ch
        x = self.pool(x)               # 112×112
        x = self.relu(self.conv2(x))   # 112×112, 32ch
        x = self.pool(x)               # 56×56
        x = self.global_max_pool(x)    # (B, 32, 1, 1)
        return x.view(x.shape[0], -1)  # (B, 32)


# ─────────────────────────────────────────────────────────────────────────────
# MainModel (CCAL) : full paper model
# concat(CNN1_out, CNN2_out) [288] → project → tokens → MHA×2 → GAP → MLP → logits
# Paper: Section III-C,D, Figures 4-6, Table 5 (heads=4, key_dim=16, value_dim=64)
# Returns: (logits, (attn_w1, attn_w2))
# ─────────────────────────────────────────────────────────────────────────────
class MainModel(nn.Module):
    """CCAL: Concatenated CNN Attention Lung cancer model.
    Paper architecture with embed_dim = heads * key_dim = 4 * 16 = 64."""
    def __init__(self, num_classes=NUM_CLASSES, num_heads=4, key_dim=16,
                 num_tokens=4, dropout_p=0.3):
        super().__init__()
        self.cnn1 = CNN_Model1(inp_chn=3)   # → (B, 256, 1, 1)
        self.cnn2 = CNN_Model2(inp_chn=3)   # → (B, 32)
        concat_dim = 256 + 32               # 288

        # Paper: embed_dim = num_heads * key_dim = 4 * 16 = 64
        embed_dim = num_heads * key_dim
        self.embed_dim  = embed_dim
        self.num_tokens = num_tokens

        # Project 288 → T*E, then reshape to (B, T, E) for MHA
        self.feature_projection = nn.Linear(concat_dim, num_tokens * embed_dim)

        # Two stacked MHA layers (paper Section III-D, Figure 5)
        self.mha1 = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads,
                                          dropout=0.1, batch_first=True)
        self.mha2 = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads,
                                          dropout=0.1, batch_first=True)

        # MLP head (paper Section III-C, Figure 6):
        # GAP1D → BN → Dense(256) → GELU → BN → Dropout → Dense(num_classes)
        self.global_avg_pool = nn.AdaptiveAvgPool1d(1)
        self.batch_norm1 = nn.BatchNorm1d(embed_dim)
        self.dense1      = nn.Linear(embed_dim, 256)
        self.gelu        = nn.GELU()
        self.batch_norm2 = nn.BatchNorm1d(256)
        self.dropout     = nn.Dropout(dropout_p)
        self.ffn         = nn.Linear(256, num_classes)

    def forward(self, x):
        f1 = self.cnn1(x).flatten(1)                                 # (B, 256)
        f2 = self.cnn2(x)                                            # (B, 32)
        concat_features = torch.cat([f1, f2], dim=1)                 # (B, 288)

        proj   = self.feature_projection(concat_features)            # (B, T*E)
        tokens = proj.view(proj.size(0), self.num_tokens, self.embed_dim)  # (B,T,E)

        attn_out1, attn_w1 = self.mha1(tokens, tokens, tokens)       # (B,T,E)
        attn_out2, attn_w2 = self.mha2(attn_out1, attn_out1, attn_out1)  # (B,T,E)

        out = attn_out2.transpose(1, 2)                              # (B,E,T)
        out = self.global_avg_pool(out).squeeze(-1)                  # (B,E)

        out = self.batch_norm1(out)
        out = self.dense1(out)
        out = self.gelu(out)
        out = self.batch_norm2(out)
        out = self.dropout(out)
        logits = self.ffn(out)
        return logits, (attn_w1, attn_w2)


# ── Sanity check ─────────────────────────────────────────────────────────────
_m = MainModel()
_n = sum(p.numel() for p in _m.parameters())
_logits, _attn = _m(torch.randn(2, 3, 224, 224))
print(f"✅ MainModel | params={_n:,} ({_n/1e6:.2f}M) | logits={_logits.shape} | attn_w1={_attn[0].shape}")
print(f"   Paper claims 14.27M — actual architecture yields {_n/1e6:.2f}M (see discrepancy analysis)")
del _m, _n, _logits, _attn

---
## 2 · Dataset & DataLoader — Paper Preprocessing

**Paper preprocessing pipeline (Section III-B):**
1. CLAHE contrast enhancement (clip_limit=2.0, tile_grid_size=4×4)
2. Resize to 224×224
3. Convert to numpy array, scale to [0, 1] by dividing by 255
4. Data augmentation: rotation, width/height shift, shear, zoom, horizontal flip
5. 80/20 train/test split (slice-level random — causes leakage)

In [ ]:
# ── CLAHE transform (paper Section III-B, Figure 3) ──────────────────────────
class ApplyCLAHE:
    """CLAHE contrast enhancement applied per-channel on an RGB PIL image.
    Paper: clip_limit=2.0, tile_grid_size=(4,4), applied before augmentation."""
    def __init__(self, clip_limit=2.0, tile_grid_size=(4, 4)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit,
                                     tileGridSize=tile_grid_size)
    def __call__(self, img):
        img_np = np.array(img)  # H × W × 3
        for i in range(3):
            img_np[:, :, i] = self.clahe.apply(img_np[:, :, i])
        return Image.fromarray(img_np)


# ── Record builder ───────────────────────────────────────────────────────────
def extract_case_id(filename):
    m = re.search(r'\((\d+)\)', filename)
    return int(m.group(1)) if m else abs(hash(filename)) % 100_000


def build_case_registry(root=DATASET_ROOT):
    root = Path(root)
    records = []
    for cls_folder in CLASS_FOLDERS:
        d = root / cls_folder
        if not d.exists():
            print(f"⚠️  Not found: {d}")
            continue
        lbl = CLASS_TO_LABEL[cls_folder]
        for f in sorted(d.iterdir()):
            if f.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}:
                records.append({
                    'image_path': str(f), 'class_name': cls_folder,
                    'label': lbl, 'case_id': extract_case_id(f.name),
                    'filename': f.name
                })
    return records


# ── Slice-level random split (paper's approach — causes leakage) ─────────────
def slice_level_split(records, seed=42):
    """Random 80/10/10 split ignoring patient boundaries.
    This is the LEAKY split that the paper uses."""
    rng = random.Random(seed)
    shuffled = list(records)
    rng.shuffle(shuffled)
    n = len(shuffled)
    n_tr = int(n * TRAIN_RATIO)
    n_v  = int(n * VAL_RATIO)
    out = []
    for i, r in enumerate(shuffled):
        r2 = dict(r)
        r2['split'] = 'train' if i < n_tr else ('val' if i < n_tr + n_v else 'test')
        out.append(r2)
    return out


def get_class_weights(records):
    train = [r for r in records if r['split'] == 'train']
    cnt   = Counter(r['label'] for r in train)
    total = sum(cnt.values())
    w = torch.zeros(NUM_CLASSES)
    for lbl, c in cnt.items():
        w[lbl] = total / (NUM_CLASSES * c)
    return w


# ── Transforms — paper-matched (Section III-B) ───────────────────────────────
def get_transforms(split, augment=False):
    """Paper preprocessing: CLAHE → Resize → augmentation → [0,1] normalize.
    Note: RGB input (3 channels), not grayscale."""
    clahe = ApplyCLAHE(clip_limit=2.0, tile_grid_size=(4, 4))
    base  = [T.Resize((IMG_SIZE, IMG_SIZE)), clahe]

    # Paper augmentation: rotation, width/height shift, shear, zoom, horizontal flip
    aug = (
        [
            T.RandomHorizontalFlip(0.5),
            T.RandomRotation(degrees=20),
            T.RandomAffine(degrees=0, translate=(0.1, 0.1),
                           shear=10, scale=(0.9, 1.1)),
        ]
        if split == 'train' and augment else []
    )

    # Paper: normalize to [0,1] by dividing by 255 — ToTensor() does this automatically
    return T.Compose(base + aug + [T.ToTensor()])


# ── PyTorch Dataset ──────────────────────────────────────────────────────────
class CTDataset(Dataset):
    def __init__(self, records, transform):
        self.records   = records
        self.transform = transform
    def __len__(self):  return len(self.records)
    def __getitem__(self, i):
        r   = self.records[i]
        img = Image.open(r['image_path']).convert('RGB')  # RGB as paper
        return self.transform(img), r['label']


def make_loaders(records, batch_size=BATCH_SIZE, augment_train=False):
    loaders = {}
    for split in ['train', 'val', 'test']:
        sub  = [r for r in records if r['split'] == split]
        tf   = get_transforms(split, augment=(split == 'train' and augment_train))
        ds   = CTDataset(sub, tf)
        loaders[split] = DataLoader(
            ds, batch_size=batch_size,
            shuffle=(split == 'train'),
            num_workers=NUM_WORKERS, pin_memory=True
        )
    return loaders['train'], loaders['val'], loaders['test']


print("✅ Dataset helpers defined (RGB + CLAHE + slice-level split).")

---
## 3 · Training Engine

In [ ]:
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR


def train_one_epoch(model, loader, optimizer, criterion, device, grad_clip):
    model.train()
    loss_sum = correct = total = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        out = model(imgs)
        logits = out[0] if isinstance(out, tuple) else out
        loss = criterion(logits, labels)
        loss.backward()
        if grad_clip:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (logits.argmax(1) == labels).sum().item()
        total    += imgs.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    loss_sum = correct = total = 0
    all_preds, all_labels, all_probs = [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        logits = out[0] if isinstance(out, tuple) else out
        loss = criterion(logits, labels)
        probs = F.softmax(logits, dim=1)
        preds = logits.argmax(1)
        loss_sum += loss.item() * imgs.size(0)
        correct  += (preds == labels).sum().item()
        total    += imgs.size(0)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        all_probs.extend(probs.cpu().tolist())
    return loss_sum / total, correct / total, all_preds, all_labels, all_probs


def compute_metrics(preds, labels, probs):
    names = [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)]
    report = classification_report(labels, preds, target_names=names,
                                    output_dict=True, zero_division=0)
    try:
        auc_val = roc_auc_score(labels, probs, multi_class='ovr', average='macro')
    except Exception:
        auc_val = float('nan')
    metrics = {
        'accuracy' : report['accuracy'],
        'f1_macro' : report['macro avg']['f1-score'],
        'auc_macro': auc_val,
        'per_class': {n: report[n] for n in names},
    }
    return metrics


def plot_curves(train_losses, val_losses, train_accs, val_accs, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(train_losses, label='Train'); axes[0].plot(val_losses, label='Val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(train_accs, label='Train'); axes[1].plot(val_accs, label='Val')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True)
    plt.tight_layout(); plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()


def plot_confusion(preds, labels, save_path, title='Confusion Matrix'):
    names = [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)]
    cm = confusion_matrix(labels, preds)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(names, rotation=30, ha='right')
    ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(names)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(title); plt.colorbar(im); plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight'); plt.show()


def plot_roc(labels, probs, save_path, title='ROC Curves'):
    """Per-class ROC curves."""
    names = [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)]
    y_bin = label_binarize(labels, classes=list(range(NUM_CLASSES)))
    probs_arr = np.array(probs)
    fig, ax = plt.subplots(figsize=(6, 5))
    for i, name in enumerate(names):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], probs_arr[:, i])
        roc_auc_val = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{name} (AUC={roc_auc_val:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
    ax.set(title=title, xlabel='FPR', ylabel='TPR')
    ax.legend(); plt.tight_layout()
    plt.savefig(save_path, dpi=100, bbox_inches='tight'); plt.show()


def run_experiment(model, train_loader, val_loader, test_loader,
                   save_dir, model_name, class_weights,
                   epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY,
                   grad_clip=GRAD_CLIP, warmup_epochs=WARMUP_EPOCHS,
                   early_stop_pat=EARLY_STOP_PAT, device=DEVICE,
                   min_delta=1e-3, smoke_test=False):
    """Full train → evaluate pipeline. Paper-matched settings:
    Adam optimizer, lr=1e-3, no weight decay, 50 epochs."""
    save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)
    if smoke_test:
        epochs = 2

    model = model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"\n🏗  {model_name} | {n_params/1e6:.2f}M params | {epochs} epochs")

    # Paper: sparse categorical cross-entropy + Adam
    crit  = nn.CrossEntropyLoss(weight=class_weights.to(device))
    opt   = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)  # Paper: Adam
    sched = CosineAnnealingLR(opt, T_max=epochs, eta_min=lr * 0.01)

    best_f1, patience_cnt = 0.0, 0
    train_losses, val_losses, train_accs, val_accs = [], [], [], []

    for epoch in range(epochs):
        t0 = time.time()
        # Warmup (paper doesn't use, but kept for compatibility)
        if epoch < warmup_epochs:
            for pg in opt.param_groups:
                pg['lr'] = lr * (epoch + 1) / warmup_epochs

        tr_loss, tr_acc = train_one_epoch(model, train_loader, opt, crit, device, grad_clip)
        vl_loss, vl_acc, vl_preds, vl_labels, vl_probs = evaluate(model, val_loader, crit, device)
        val_m = compute_metrics(vl_preds, vl_labels, vl_probs)
        sched.step()

        train_losses.append(tr_loss); val_losses.append(vl_loss)
        train_accs.append(tr_acc);   val_accs.append(vl_acc)

        marker = ''
        if val_m['f1_macro'] > best_f1 + min_delta:
            best_f1 = val_m['f1_macro']
            torch.save(model.state_dict(), save_dir / f"{model_name}_best.pth")
            patience_cnt = 0
            marker = ' ★'
        else:
            patience_cnt += 1
            if patience_cnt >= early_stop_pat:
                print(f"  Early stopping at epoch {epoch+1}.")
                break

        print(f"  [{epoch+1:3d}/{epochs}] "
              f"tr={tr_loss:.4f}/{tr_acc:.4f} | "
              f"vl={vl_loss:.4f}/{vl_acc:.4f} "
              f"f1={val_m['f1_macro']:.4f} "
              f"({time.time()-t0:.1f}s){marker}")

    # ── Final test evaluation ─────────────────────────────────────────────────
    ckpt = save_dir / f"{model_name}_best.pth"
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location=device))
    _, _, te_preds, te_labels, te_probs = evaluate(model, test_loader, crit, device)
    test_m = compute_metrics(te_preds, te_labels, te_probs)
    test_m['params_M'] = round(n_params / 1e6, 3)

    # Plots
    plot_curves(train_losses, val_losses, train_accs, val_accs,
                save_dir / f"{model_name}_curves.png")
    plot_confusion(te_preds, te_labels,
                   save_dir / f"{model_name}_confusion.png", title=model_name)
    plot_roc(te_labels, te_probs,
             save_dir / f"{model_name}_roc.png", title=f"{model_name} — ROC")

    # Print report
    names = [LABEL_TO_CLASS[i] for i in range(NUM_CLASSES)]
    print("\n" + "=" * 60)
    print(f"  TEST — {model_name}")
    print("=" * 60)
    print(classification_report(te_labels, te_preds, target_names=names, zero_division=0))

    # Save metrics
    with open(save_dir / "metrics.json", 'w') as f:
        json.dump(test_m, f, indent=2)

    return test_m


print("✅ Training engine defined (Adam, paper-matched).")

---
## 4 · Run Experiment 1A — Slice-Level Split (Leakage Demo)

> **Expected:** ~95–99% accuracy — this is the **inflated** result caused by data leakage.
>
> Slices from the same patient appear in both train and test sets.

In [ ]:
print("=" * 68)
print(" EXPERIMENT 1A — Slice-Level Random Split (Leakage Demo)")
print("   Paper: 80/20 split, CLAHE, RGB, Adam lr=1e-3")
print("   Expected: ~95–99% accuracy — this is the INFLATED result")
print("=" * 68)

records_all   = build_case_registry()
records_slice = slice_level_split(records_all)

print(f"\nTotal images: {len(records_all)}")
print(f"Total cases:  {len(set(r['case_id'] for r in records_all))}")

print("\nSplit distribution (slices — leaky!):")
for split in ['train', 'val', 'test']:
    sub = [r for r in records_slice if r['split'] == split]
    cases = len(set(r['case_id'] for r in sub))
    print(f"  {split:<6}: {len(sub):>4} imgs, {cases:>3} cases | {dict(Counter(r['class_name'] for r in sub))}")

# Check leakage: how many cases appear in multiple splits?
case_splits = defaultdict(set)
for r in records_slice:
    case_splits[r['case_id']].add(r['split'])
leaked = {c: s for c, s in case_splits.items() if len(s) > 1}
print(f"\n🚨 Cases appearing in MULTIPLE splits: {len(leaked)} / {len(case_splits)} ({len(leaked)/len(case_splits)*100:.1f}%)")
print("   This IS the data leakage.")

In [ ]:
train_ld, val_ld, test_ld = make_loaders(records_slice, augment_train=True)
cw_slice = get_class_weights(records_slice)
print(f"Class weights: {cw_slice.tolist()}")

slice_metrics = run_experiment(
    model          = MainModel(),
    train_loader   = train_ld,
    val_loader     = val_ld,
    test_loader    = test_ld,
    save_dir       = RESULTS_DIR / "1a_slice_split",
    model_name     = "CCAL_SliceSplit",
    class_weights  = cw_slice,
    lr             = 1e-3,
    weight_decay   = 0.0,
    warmup_epochs  = 0,
    early_stop_pat = 999,  # train all 50 epochs like paper
)

print(f"\n📊 1A Slice Split — Test Accuracy : {slice_metrics['accuracy']*100:.2f}%")
print(f"   F1-Macro                        : {slice_metrics['f1_macro']:.4f}")
print(f"   AUC-Macro                       : {slice_metrics['auc_macro']:.4f}")
print(f"   Benign Recall                   : {slice_metrics['per_class']['Benign']['recall']:.4f}")

---
## 5 · Summary

This experiment reproduces the paper's exact methodology and should yield ~95-99% accuracy,
matching their claimed 99.54%. The inflated result is caused by **slice-level data leakage**:
CT slices from the same patient appear in both training and test sets.

→ Compare with **Experiment 1B** (sequential split) to see the true performance drop when leakage is eliminated.

In [ ]:
print("\n" + "=" * 70)
print(" EXPERIMENT 1A — SUMMARY")
print("=" * 70)
print(f"  Model        : CCAL (paper architecture, {slice_metrics.get('params_M', '?')}M params)")
print(f"  Preprocessing: RGB + CLAHE + [0,1] normalization")
print(f"  Split        : 80/10/10 slice-level random (LEAKY)")
print(f"  Optimizer    : Adam, lr=1e-3, no weight decay")
print(f"  ─────────────────────────────")
print(f"  Test Accuracy: {slice_metrics['accuracy']*100:.2f}%")
print(f"  F1-Macro     : {slice_metrics['f1_macro']:.4f}")
print(f"  AUC-Macro    : {slice_metrics['auc_macro']:.4f}")
print(f"  Benign Recall: {slice_metrics['per_class']['Benign']['recall']:.4f}")
print("=" * 70)
print("\n⚠️  These results are INFLATED due to data leakage.")
print("   See Experiment 1B for honest evaluation.")